# SciGraph Repair: Reproducible Synthetic Pilot

**Graph-linked scientific invariants for local software repair tools.** This notebook reconstructs the source code in its own working directory, runs deterministic original-source tests, and reproduces the reported mutation and retrieval statistics.

**Scope caution:** No Gemma 4 model is loaded, trained, or evaluated here. The 24 seeded mutants are deliberately authored and tests are written with knowledge of their types. These results do not establish real-world agent performance. No personal health data are used.

The prototype uses the Python standard library. It builds an AST graph without executing scanned repository source; only the controlled benchmark's fixture source is executed in the mutation audit.

In [ ]:
from pathlib import Path
for d in ["scigraph", "benchmark/fixture_repo", "results", "tests"]:
    Path(d).mkdir(parents=True, exist_ok=True)
print("Created working directories")

### scigraph/__init__.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile scigraph/__init__.py
"""SciGraph Repair: read-only AST context builder for scientific-code review."""
from .graph import inspect_repository, build_context
__all__ = ['inspect_repository', 'build_context']


### scigraph/graph.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile scigraph/graph.py
"""Standard-library-only repository-graph retrieval. Does not execute target code."""
from __future__ import annotations
from collections import Counter, defaultdict
from pathlib import Path
import ast
import json
import math
import re

STOP = set('a an the is are of for in it to with on from and or that this when after before than into by be i we should fix bug incorrect wrong value data code issue error results unexpected function reports report gives returning return'.split())

def tokens(s: str):
    s = re.sub(r'([a-z])([A-Z])', r'\1 \2', s)
    return [w for w in re.findall(r'[a-z0-9]+', s.lower().replace('_',' ')) if w not in STOP and len(w) >= 2]

def inspect_repository(root: str | Path):
    """Parse .py files in root (read only), returning nodes & static call edges."""
    root = Path(root).resolve()
    nodes = {}; calls = defaultdict(list)
    for path in sorted(root.rglob('*.py')):
        if any(x in path.parts for x in ('.git','.venv','venv','__pycache__')):
            continue
        try:
            source = path.read_text(encoding='utf-8')
            tree = ast.parse(source, filename=str(path))
        except (UnicodeError, SyntaxError, OSError):
            continue
        rel = path.relative_to(root).as_posix()
        for unit in tree.body:
            funcs = []
            if isinstance(unit, (ast.FunctionDef, ast.AsyncFunctionDef)):
                funcs = [(unit,unit.name)]
            elif isinstance(unit,ast.ClassDef):
                funcs = [(member, unit.name+'.'+member.name) for member in unit.body if isinstance(member,(ast.FunctionDef,ast.AsyncFunctionDef))]
            for fn, name in funcs:
                id_ = f'{rel}::{name}'
                snip = ast.get_source_segment(source,fn) or ''
                call_names=[]
                for n in ast.walk(fn):
                    if isinstance(n,ast.Call):
                        if isinstance(n.func,ast.Name): call_names.append(n.func.id)
                        elif isinstance(n.func,ast.Attribute): call_names.append(n.func.attr)
                nodes[id_] = {'id':id_,'file':rel,'name':name,'line':fn.lineno,
                              'source':snip,'doc':ast.get_docstring(fn) or '', 'call_names':call_names}
    by_short=defaultdict(list)
    for nid,n in nodes.items(): by_short[n['name'].split('.')[-1]].append(nid)
    edges=[]
    for nid,n in nodes.items():
        for short in n['call_names']:
            for target in by_short.get(short,[]):
                if target != nid:
                    edges.append({'source':nid,'target':target,'relation':'calls'})
    edges=sorted({(e['source'],e['target']) for e in edges})
    return {'root':str(root),'nodes':list(nodes.values()),
            'edges':[{'source':a,'target':b,'relation':'calls'} for a,b in edges]}

def _idf(n, query_words, document_count):
    return sum((1 + math.log((document_count+1)/(1+n.get(w,0)))) for w in query_words if w in n)

def rank_nodes(graph, issue, mode='graph', budget_chars=4500):
    """Contrast lexical retrieval with local dependency propagation."""
    nodes={n['id']:n for n in graph['nodes']}
    if not nodes: return []
    query=set(tokens(issue)); corpus={}
    for k,node in nodes.items():
        words=tokens(node['file']+' '+node['name']+' '+node['doc']+' '+node['source'])
        corpus[k]=Counter(words)
    df=Counter(w for cnt in corpus.values() for w in cnt)
    N=len(nodes)
    base={}
    for k,cnt in corpus.items():
        # Weighted token relevance plus exact symbol-name cues. Absolute scale non-essential.
        base[k] = sum((1+math.log(cnt[w])) * math.log(1+(N+1)/(df[w]+1)) for w in query if cnt[w] > 0)
        label=set(tokens(nodes[k]['name']+' '+nodes[k]['file']))
        base[k]+=2.2*len(query.intersection(label))
    vals=dict(base)
    if mode=='graph':
        neighborhood=defaultdict(set)
        for edge in graph['edges']:
            u,v=edge['source'],edge['target']; neighborhood[u].add(v); neighborhood[v].add(u)
        for _ in range(2):
            vals={n:0.80*base[n]+0.20*sum(vals.get(nb,0)/max(1,len(neighborhood[nb])) for nb in neighborhood[n]) for n in nodes}
    order=sorted(nodes,key=lambda k:(-vals[k],k))
    chars=0; selected=[]
    for k in order:
        n=nodes[k]; footprint=min(len(n['source']),1100)+len(k)+80
        if selected and chars+footprint>budget_chars: continue
        if footprint>budget_chars and not selected: footprint=budget_chars
        selected.append({**n,'score':round(vals[k],5),'snippet':n['source'][:max(100,min(1100,budget_chars-chars))]})
        chars+=footprint
    return selected

def build_context(root, issue, mode='graph', budget_chars=4500):
    g=inspect_repository(root)
    chosen=rank_nodes(g,issue,mode,budget_chars)
    selected=set(n['id'] for n in chosen)
    relevant_edges=[e for e in g['edges'] if e['source'] in selected and e['target'] in selected]
    result={'issue':issue,'mode':mode,'character_budget':budget_chars,
            'total_symbols':len(g['nodes']),'total_edges':len(g['edges']),
            'selected_symbols':chosen,'selected_edges':relevant_edges,
            'disclaimer':'Static AST edges are approximations; dynamic dispatch and aliasing are not fully resolved. No bug fix is generated.'}
    from .contracts import select_contracts
    result['verification_candidates']=select_contracts(result)
    return result


### scigraph/contracts.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile scigraph/contracts.py
"""Human-authored scientific contracts for optional code-review assistance.

These assertions are suggestions, not a proof of correctness and not universally valid
for all public health datasets. Apply only after validating the statistical definitions.
"""
CONTRACTS={
 'vaccine_coverage': {
    'meaning':'Vaccinated / eligible, represented as a proportion',
    'checks':['Check numerator and denominator represent the same cohort and period',
              'Under ideal consistent counts, 0 <= coverage <= 1; values over 1 can also reflect reporting artifacts',
              'Joint multiplication of counts should preserve the proportion'],
    'example':'assert abs(vaccine_coverage(40,80) - 0.5) < 1e-10'},
 'dropout_rate': {
    'meaning':'(First dose - final dose) / first dose',
    'checks':['Check dose order and denominator','Negative dropout can occur in reported aggregate data',
              'Scaling dose counts together should preserve dropout'],
    'example':'assert abs(dropout_rate(80,60) - 0.25) < 1e-10'},
 'incidence_per_100k': {
    'meaning':'Case count / population * 100,000',
    'checks':['Confirm the scale is per 100,000','Verify person-time or population denominator is appropriate',
              'Joint multiplication of counts and population should preserve rate'],
    'example':'assert abs(incidence_per_100k(5,10000) - 50) < 1e-10'},
 'rolling_mean_past': {
    'meaning':'Trailing window average without future information',
    'checks':['Prefix outputs should not change after future values are appended',
              'Constant inputs should yield constant outputs','Check first-window convention'],
    'example':'assert rolling_mean_past([1,3,9,2],2) == [1,2,6,5.5]'},
 'weighted_rate': {
    'meaning':'Pooled rate equals total events / total denominator',
    'checks':['Do not substitute unweighted mean of stratum rates',
              'Shuffling strata should not change the pooled rate',
              'Joint rescaling of all strata should not change the pooled rate'],
    'example':'assert abs(weighted_rate([(1,10),(80,100)]) - 81/110) < 1e-10'},
 'deduplicate_records': {
    'meaning':'Retain first instance per person identifier',
    'checks':['Check unique identifier semantics across sites',
              'Verify retained-record precedence is as specified',
              'Confirm de-duplication does not silently discard records with incomplete IDs'],
    'example':"assert len(deduplicate_records([{'id':'A'},{'id':'A'}])) == 1"},
 'positivity_rate': {
    'meaning':'Positive test results / number tested',
    'checks':['Numerator should not exceed denominator for coherent linked person counts',
              'Zero positives should produce zero when denominator positive',
              'Counts must refer to the same interval and test-definition'],
    'example':'assert abs(positivity_rate(10,40) - .25) < 1e-10'},
 'cumulative_cases': {
    'meaning':'Running sum of incident counts in chronological order',
    'checks':['For nonnegative incident counts, cumulative output should not decrease',
              'Appending new observations should not change earlier cumulative totals',
              'A zero-case period should preserve the previous cumulative total'],
    'example':'assert cumulative_cases([2,3,1]) == [2,5,6]'},
}

def select_contracts(context, max_contracts=8):
    out=[]
    for n in context.get('selected_symbols',[]):
        name=n['name'].split('.')[-1]
        if name in CONTRACTS:
            out.append({'symbol':n['id'],'line':n['line'],**CONTRACTS[name]})
    return out[:max_contracts]


### scigraph/cli.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile scigraph/cli.py
"""CLI for read-only code repository context generation."""
import argparse,json
from .graph import build_context

def main():
    p=argparse.ArgumentParser(description='Build a budgeted code-graph context pack for small offline coding agents')
    p.add_argument('repository',help='Python repository directory (read only)')
    p.add_argument('--issue',required=True,help='Issue description')
    p.add_argument('--mode',choices=['lexical','graph'],default='graph')
    p.add_argument('--chars',type=int,default=4500)
    args=p.parse_args()
    print(json.dumps(build_context(args.repository,args.issue,args.mode,args.chars),indent=2))

if __name__=='__main__':main()


### benchmark/__init__.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile benchmark/__init__.py
"""Controlled synthetic benchmark; contains no real-world patient data."""


### benchmark/fixture_repo/epi_analytics.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile benchmark/fixture_repo/epi_analytics.py
"""Original, deliberately small epidemiologic indicator calculations for a controlled study.

No real patient records or third-party repository code is included.
"""

def vaccine_coverage(vaccinated, eligible):
    """Proportion eligible receiving a vaccine."""
    return vaccinated / eligible

def dropout_rate(first_dose, final_dose):
    """DTP1-to-DTP3 dropout ratio."""
    return (first_dose - final_dose) / first_dose

def incidence_per_100k(cases, population):
    """Cases per one hundred thousand population."""
    return cases / population * 100000

def rolling_mean_past(values, window):
    """Trailing mean uses current and historical periods only."""
    out = []
    for i in range(len(values)):
        window_values = values[max(0, i - window + 1):i + 1]
        out.append(sum(window_values) / len(window_values))
    return out

def weighted_rate(group_counts):
    """Pooled rate from (events, population) pairs, not mean of stratum rates."""
    return sum(n for n, d in group_counts) / sum(d for n, d in group_counts)

def deduplicate_records(records):
    """Keep first record for each person identifier."""
    seen = set()
    output = []
    for rec in records:
        if rec['id'] not in seen:
            seen.add(rec['id'])
            output.append(rec)
    return output

def positivity_rate(positive, tested):
    """Positive laboratory results over all people tested."""
    return positive / tested

def cumulative_cases(values):
    """Cumulative sum, preserving time order."""
    result = []
    total = 0
    for x in values:
        total += x
        result.append(total)
    return result


### benchmark/fixture_repo/pipeline.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile benchmark/fixture_repo/pipeline.py
"""A minimal dependent analytics pipeline to illustrate graph-guided issue context."""
from epi_analytics import (vaccine_coverage, dropout_rate, incidence_per_100k,
                           rolling_mean_past, weighted_rate, deduplicate_records,
                           positivity_rate, cumulative_cases)

def immunization_dashboard(vaccinated, eligible, first_dose, final_dose):
    return {'coverage':vaccine_coverage(vaccinated, eligible),
            'dropout':dropout_rate(first_dose,final_dose)}

def outbreak_dashboard(cases, population, observations):
    return {'incidence': incidence_per_100k(cases, population),
            'trend': rolling_mean_past(observations,3),
            'cumulative': cumulative_cases(observations)}

def laboratory_dashboard(positive, tested, records):
    deduped = deduplicate_records(records)
    return {'positivity':positivity_rate(positive,tested), 'patients':len(deduped)}

def equity_dashboard(groups):
    return {'pooled_rate':weighted_rate(groups)}


### benchmark/evaluate.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile benchmark/evaluate.py
"""Deterministic, seeded mutation audit using an original synthetic epidemiology library.

This is an engineering feasibility experiment, NOT an assessment of Gemma 4 patch accuracy.
Every mutated function is executed in an isolated in-memory namespace controlled here.
Never use this pattern to load arbitrary untrusted source code.
"""
from pathlib import Path
import json, math, types, statistics, sys
from scigraph.graph import inspect_repository, rank_nodes

HERE=Path(__file__).resolve().parent
SOURCE=(HERE/'fixture_repo'/'epi_analytics.py').read_text(encoding='utf-8')
MUTANTS={
 'vaccine_coverage':[
  ('reversed_denominator','return vaccinated / eligible','return eligible / vaccinated'),
  ('offset_numerator','return vaccinated / eligible','return (vaccinated + 10) / eligible'),
  ('percent_proportion_mixup','return vaccinated / eligible','return vaccinated / eligible * 100'),
 ],
 'dropout_rate':[
  ('wrong_denominator','return (first_dose - final_dose) / first_dose','return (first_dose - final_dose) / final_dose'),
  ('retention_not_dropout','return (first_dose - final_dose) / first_dose','return final_dose / first_dose'),
  ('negative_values_hidden','return (first_dose - final_dose) / first_dose','return abs(first_dose - final_dose) / first_dose'),
 ],
 'incidence_per_100k':[
  ('per_1000_not_100k','return cases / population * 100000','return cases / population * 1000'),
  ('case_pseudocount','return cases / population * 100000','return (cases + 1) / population * 100000'),
  ('off_by_100_denominator','return cases / population * 100000','return cases / (population - 100) * 100000'),
 ],
 'rolling_mean_past':[
  ('future_window','window_values = values[max(0, i - window + 1):i + 1]','window_values = values[i:min(len(values), i + window)]'),
  ('lookahead_leakage','window_values = values[max(0, i - window + 1):i + 1]','window_values = values[max(0, i - window + 1):i + 2]'),
  ('one_period_too_short','window_values = values[max(0, i - window + 1):i + 1]','window_values = values[max(0, i - window + 2):i + 1]'),
 ],
 'weighted_rate':[
  ('unweighted_group_average','return sum(n for n, d in group_counts) / sum(d for n, d in group_counts)','return sum(n / d for n,d in group_counts) / len(group_counts)'),
  ('numerator_denominator_swapped','return sum(n for n, d in group_counts) / sum(d for n, d in group_counts)','return sum(d for n, d in group_counts) / sum(n for n, d in group_counts)'),
  ('added_denominator_offset','return sum(n for n, d in group_counts) / sum(d for n, d in group_counts)','return sum(n for n, d in group_counts) / (sum(d for n, d in group_counts)+10)'),
 ],
 'deduplicate_records':[
  ('inverted_condition',"if rec['id'] not in seen:","if rec['id'] in seen:"),
  ('compound_person_site_key',"seen.add(rec['id'])","seen.add(rec['id'] + rec['site'])"),
  ('unexpected_reverse_order','output.append(rec)','output.insert(0, rec)'),
 ],
 'positivity_rate':[
  ('reversed_denominator','return positive / tested','return tested / positive'),
  ('numerator_pseudocount','return positive / tested','return (positive + 1) / tested'),
  ('wrong_denominator_offset','return positive / tested','return positive / (tested + 10)'),
 ],
 'cumulative_cases':[
  ('reset_each_period','total += x','total = x'),
  ('count_periods_not_cases','total += x','total += 1'),
  ('subtract_cases','total += x','total -= x'),
 ]
}

ISSUES=[
 ('immunization_dashboard results show vaccine coverage too high for eligible population','vaccine_coverage'),
 ('immunization_dashboard drop-off wrong even when dose counts are inconsistent','dropout_rate'),
 ('outbreak_dashboard reports a suspiciously low disease burden','incidence_per_100k'),
 ('outbreak_dashboard trend changes when newer periods are appended','rolling_mean_past'),
 ('equity_dashboard changes with different population sizes across strata','weighted_rate'),
 ('laboratory_dashboard patient count inflated by cross-site records','deduplicate_records'),
 ('laboratory_dashboard infection positivity appears implausible','positivity_rate'),
 ('outbreak_dashboard running total decreases after reporting new cases','cumulative_cases'),
]

def approx(a,b): return math.isclose(a,b,rel_tol=1e-9,abs_tol=1e-9)

def example_records():
    return [{'id':'A','site':'X','stamp':1},{'id':'B','site':'Y','stamp':2},{'id':'A','site':'Z','stamp':3}]

def smoke_ok(m, fn):
    try:
        args={
            'vaccine_coverage':(40,80), 'dropout_rate':(80,60),
            'incidence_per_100k':(5,10000), 'rolling_mean_past':([1,3,9,2],2),
            'weighted_rate':([(1,10),(80,100)],),
            'deduplicate_records':(example_records(),),
            'positivity_rate':(10,40),'cumulative_cases':([2,3,1],)
        }[fn]
        value=getattr(m,fn)(*args)
        if isinstance(value, list):
            return all(isinstance(x,(int,float,dict)) for x in value)
        return isinstance(value,(int,float)) and math.isfinite(value)
    except Exception: return False

def invariant_ok(m, fn):
    """Non-oracle metamorphic checks; does not supply exact expected clinical values."""
    try:
        f=getattr(m,fn)
        if fn=='vaccine_coverage':
            return approx(f(40,80),f(80,160)) and approx(f(0,80),0) and 0<=f(40,80)<=1
        if fn=='dropout_rate':
            return approx(f(80,60),f(160,120)) and f(80,100)<0 and f(80,60)>0
        if fn=='incidence_per_100k':
            return approx(f(5,10000),f(10,20000)) and approx(f(0,10000),0)
        if fn=='rolling_mean_past':
            a=f([1,3,9,2],2); b=f([1,3,9,2,1000],2)
            return len(a)==4 and all(approx(x,y) for x,y in zip(a,b[:4])) and approx(f([7]*5,3)[-1],7)
        if fn=='weighted_rate':
            pairs=[(1,10),(80,100)]
            return approx(f(pairs),f([(n*2,d*2) for n,d in pairs])) and approx(f(pairs),f(list(reversed(pairs))))
        if fn=='deduplicate_records':
            a=f(example_records())
            return len(a)==2 and len({x['id'] for x in a})==2 and a[0]['stamp']==1
        if fn=='positivity_rate':
            return approx(f(10,40),f(20,80)) and approx(f(0,40),0) and 0<=f(10,40)<=1
        if fn=='cumulative_cases':
            a=f([2,3,1]); b=f([2,3,1,9]);return len(a)==3 and a==b[:3] and a==sorted(a)
    except Exception: return False
    return False

def semantic_ok(m, fn):
    """Domain anchors + invariants. These held-in-code values constitute a seeded test oracle."""
    try:
        f=getattr(m,fn)
        anchors={
         'vaccine_coverage':lambda: approx(f(40,80),.5) and approx(f(75,100),.75),
         'dropout_rate':lambda: approx(f(80,60),.25) and approx(f(80,100),-.25),
         'incidence_per_100k':lambda: approx(f(5,10000),50) and approx(f(0,20000),0),
         'rolling_mean_past':lambda: all(approx(x,y) for x,y in zip(f([1,3,9,2],2),[1,2,6,5.5])),
         'weighted_rate':lambda: approx(f([(1,10),(80,100)]),81/110) and approx(f([(0,100),(2,2)]),2/102),
         'deduplicate_records':lambda: [rec['id'] for rec in f(example_records())]==['A','B'],
         'positivity_rate':lambda: approx(f(10,40),.25) and approx(f(0,40),0),
         'cumulative_cases':lambda: f([2,3,1])==[2,5,6]
        }
        return invariant_ok(m,fn) and anchors[fn]()
    except Exception: return False

def load_source(text):
    m=types.ModuleType('controlled_seeded_epi_analytics')
    exec(compile(text,'controlled_seeded_epi_analytics.py','exec'),m.__dict__)
    return m

def wilson(successes,total,z=1.96):
    if total==0: return [None,None]
    p=successes/total; den=1+z*z/total
    center=(p+z*z/(2*total))/den
    half=z*math.sqrt(p*(1-p)/total+z*z/(4*total*total))/den
    return [round(center-half,4),round(center+half,4)]

def main():
    control=load_source(SOURCE)
    control_results={fn:{'smoke_pass':smoke_ok(control,fn),'invariant_pass':invariant_ok(control,fn),'semantic_pass':semantic_ok(control,fn)} for fn in MUTANTS}
    assert all(all(v.values()) for v in control_results.values()),control_results
    records=[]
    for fn, variants in MUTANTS.items():
        for label,old,new in variants:
            assert SOURCE.count(old)==1,(fn,label,SOURCE.count(old))
            mutated=load_source(SOURCE.replace(old,new,1))
            records.append({'function':fn,'mutation':label,
                'smoke_detected':not smoke_ok(mutated,fn),
                'invariant_detected':not invariant_ok(mutated,fn),
                'semantic_detected':not semantic_ok(mutated,fn)})
    graph=inspect_repository(HERE/'fixture_repo')
    retrieval=[]
    for issue,fn in ISSUES:
        for budget in (600,1200,2400):
            row={'issue':issue,'expected_function':fn,'budget_chars':budget}
            for mode in ('lexical','graph'):
                ranked=rank_nodes(graph,issue,mode,budget)
                row[mode+'_top1']=int(bool(ranked) and ranked[0]['name']==fn)
                row[mode+'_hit']=int(any(n['name']==fn for n in ranked))
                row[mode+'_symbols']=[n['name'] for n in ranked]
            retrieval.append(row)
    total=len(records)
    metrics={}
    for mode in ('smoke','invariant','semantic'):
        s=sum(r[mode+'_detected'] for r in records)
        metrics[mode]={'detected':s,'total':total,'rate':round(s/total,4),'wilson_95':wilson(s,total)}
    retrieval_metrics={}
    for budget in (600,1200,2400):
        subset=[r for r in retrieval if r['budget_chars']==budget]
        retrieval_metrics[str(budget)]={mode:{'top1':sum(r[mode+'_top1'] for r in subset),
                                                  'hit':sum(r[mode+'_hit'] for r in subset),
                                                  'tasks':len(subset)} for mode in ('lexical','graph')}
    result={'experiment':'Controlled, purpose-built synthetic audit. NOT a Gemma 4 assessment.',
            'n_original_functions':len(MUTANTS),'n_mutants':total,'n_retrieval_issues':len(ISSUES),
            'control_checks':control_results,'mutation_detection':metrics,
            'localization':retrieval_metrics,'mutants':records,'retrieval_rows':retrieval,
            'limitations':['Mutations are authored with knowledge of the test oracles, so detection estimates are optimistic.',
                'The synthetic benchmark is tiny and not representative of Kaggle hidden issues.',
                'The localizer is heuristic; no language model, Gemma weights, or patch-generation experiment was run.',
                'Static call graph cannot completely resolve Python aliases, inheritance, or dynamic dispatch.']}
    out=HERE.parent/'results'/'pilot_results.json';out.parent.mkdir(parents=True,exist_ok=True)
    out.write_text(json.dumps(result,indent=2)+'\n',encoding='utf-8')
    print(json.dumps({'mutation_detection':metrics,'localization':retrieval_metrics,'result_file':str(out)},indent=2))

if __name__=='__main__':main()


### tests/test_graph.py

The following cell creates this exact source file for reproducibility.

In [ ]:
%%writefile tests/test_graph.py
import unittest
from pathlib import Path
from scigraph.graph import inspect_repository,build_context
from benchmark.evaluate import MUTANTS, SOURCE, load_source, invariant_ok, semantic_ok

ROOT=Path(__file__).resolve().parents[1]
class TestGraph(unittest.TestCase):
    def test_graph_call_edge(self):
        g=inspect_repository(ROOT/'benchmark'/'fixture_repo')
        self.assertTrue(any('immunization_dashboard' in e['source'] and 'vaccine_coverage' in e['target'] for e in g['edges']))
    def test_read_only_context(self):
        obj=build_context(ROOT/'benchmark'/'fixture_repo','vaccine coverage dashboard error',budget_chars=1600)
        self.assertTrue(obj['selected_symbols'])
        self.assertEqual(obj['mode'],'graph')
    def test_mutation_reference(self):
        m=load_source(SOURCE)
        self.assertTrue(all(semantic_ok(m,f) for f in MUTANTS))
    def test_detect_swap(self):
        bad=SOURCE.replace('return vaccinated / eligible','return eligible / vaccinated',1)
        self.assertFalse(semantic_ok(load_source(bad),'vaccine_coverage'))
if __name__=='__main__':unittest.main()


## Run deterministic tests and mutation experiment

The initial source functions must pass every semantic check. Each deliberate mutation is then assessed by three different test tiers.

In [ ]:
import subprocess, sys
checks = subprocess.run([sys.executable,"-m","unittest","discover","-s","tests","-v"], capture_output=True, text=True)
print(checks.stdout+checks.stderr)
assert checks.returncode == 0

In [ ]:
experiment = subprocess.run([sys.executable,"-m","benchmark.evaluate"], capture_output=True, text=True)
print(experiment.stdout+experiment.stderr)
assert experiment.returncode == 0

## Inspect actual results, including controls and limitations

In [ ]:
import json
R=json.loads(Path("results/pilot_results.json").read_text())
print("Mutation detection")
for k,v in R["mutation_detection"].items():
    print(k, v["detected"], "/", v["total"], "detected; Wilson interval",v["wilson_95"])
print("\nLocalization: relevant underlying function included")
for budget, arms in R["localization"].items():
    print("budget",budget,"characters: ",", ".join(f"{arm} {x['hit']}/{x['tasks']}" for arm,x in arms.items()))
print("\nLimitations:")
for limitation in R["limitations"]:print("•",limitation)

## Example of compact, read-only context packaging

In [ ]:
from scigraph.graph import build_context
example=build_context("benchmark/fixture_repo", "immunization dashboard shows coverage above 100 percent",budget_chars=1500)
print("Symbols:",[(s["name"],round(s["score"],2)) for s in example["selected_symbols"]])
print("Proposed scientific checks:",[(c["symbol"],c["meaning"]) for c in example["verification_candidates"]])
print("No files are edited, and no patch has been generated.")

## Suggested independently validated follow-up

Use the same Gemma 4 model and compute budget under three prespecified approaches: lexical retrieval, graph retrieval, and graph retrieval plus independently written scientific contracts. Use *unseen* problems, report patch-level pass/fail and semantic correctness separately, and publish task IDs, seeds, token usage, and failures. Do not reuse these synthetic pilot scores as evidence of real-model improvement.

**Literature anchors:** [SWE-bench (ICLR 2024)](https://proceedings.iclr.cc/paper_files/paper/2024/hash/edac78c3e300629acfe6cbe9ca88fb84-Abstract-Conference.html); [RepoGraph (ICLR 2025)](https://proceedings.iclr.cc/paper_files/paper/2025/hash/4a4a3c197deac042461c677219efd36c-Abstract-Conference.html); [Agentless (FSE 2025)](https://conf.researchr.org/details/fse-2025/fse-2025-research-papers/85/Demystifying-LLM-based-Software-Engineering-Agents); [Traceability analysis (ISSTA 2026)](https://research.ibm.com/publications/understanding-automated-program-repair-agents-through-the-lens-of-traceability-an-empirical-study).